In [5]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.metrics import f1_score, classification_report

# ==========================================
# 1. SETUP PATH & LOAD DATA (TRAIN & TEST)
# ==========================================
root_path = Path.cwd().parent

print("⏳ 1. Memuat Data Latih, Data Uji, dan Model Final...")
# Mengambil data latih yang sudah diseleksi fiturnya
train_df = pd.read_csv(root_path / "Data/processed/train_selected_features.csv")

# ✅ PERUBAHAN: Mengambil data uji dari folder split
test_df = pd.read_csv(root_path / "Data/split/test_data.csv") 

selected_features = ['TIPI1', 'TIPI2', 'TIPI3', 'TIPI4', 'TIPI5', 'TIPI6', 'TIPI7', 'TIPI8', 'TIPI9', 'TIPI10', 
                     'VCL1', 'VCL2', 'VCL3', 'VCL4', 'VCL5', 'VCL7', 'VCL8', 'VCL9', 'VCL10', 'VCL11', 'VCL12', 
                     'VCL13', 'VCL14', 'VCL15', 'VCL16', 'education', 'urban', 'gender', 'engnat', 'age', 
                     'religion', 'orientation', 'race', 'voted', 'married']
target_cols = ['risk_depression', 'risk_anxiety', 'risk_stress']

X_train = train_df[selected_features]
Y_train = train_df[target_cols]

# Kode ini akan otomatis hanya mengambil 35 kolom dari test_data.csv
X_test = test_df[selected_features] 
Y_test = test_df[target_cols]

# ==========================================
# 2. LOAD MODEL FINAL (Versi Terbaikmu)
# ==========================================
model_path = root_path / "models/multilabel_xgboost_mfo.pkl"
with open(model_path, "rb") as f:
    final_model = pickle.load(f)

# ==========================================
# 3. CARI THRESHOLD MENGGUNAKAN DATA LATIH (HALAL & VALID)
# ==========================================
print("\n🔍 2. Mencari Threshold Terbaik Menggunakan Data Latih (X_train)...")
y_train_proba = final_model.predict_proba(X_train)
best_thresholds = []

for i, target_name in enumerate(target_cols):
    best_t = 0.5
    best_f1 = 0.0
    
    # PERBAIKAN: Ambil probabilitas kelas 1 (Berisiko) dari target ke-i
    prob_train_positive = y_train_proba[i][:, 1]
    
    for t in np.arange(0.1, 0.9, 0.01):
        y_pred_temp = (prob_train_positive >= t).astype(int)
        current_f1 = f1_score(Y_train.iloc[:, i], y_pred_temp)
        
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_t = t
            
    best_thresholds.append(best_t)
    print(f"   -> {target_name.upper()}: Threshold Kunci = {best_t:.2f} (Estimasi F1 Train: {best_f1:.4f})")

# ==========================================
# 4. EVALUASI DI DATA TESTING MURNI (PEMBUKTIAN)
# ==========================================
print("\n🏆 3. Menerapkan Threshold Kunci pada Data Uji Murni (X_test) 🏆")
y_test_proba = final_model.predict_proba(X_test)

# Siapkan kanvas kosong untuk tebakan final
y_test_optimized = np.zeros((len(X_test), len(target_cols)))

for i in range(len(target_cols)):
    # PERBAIKAN: Ambil probabilitas kelas 1 untuk data testing
    prob_test_positive = y_test_proba[i][:, 1]
    
    # Gunakan batas threshold yang didapat dari data latih tadi
    y_test_optimized[:, i] = (prob_test_positive >= best_thresholds[i]).astype(int)

print("-" * 50)
macro_f1 = f1_score(Y_test, y_test_optimized, average='macro')
print(f"🌟 MACRO F1-SCORE FINAL (LEGAL): {macro_f1:.4f}")
print("-" * 50)
print(classification_report(Y_test, y_test_optimized, target_names=['Depression', 'Anxiety', 'Stress']))
# # ==========================================
# # 5. EVALUASI AKHIR SETELAH OPTIMASI
# # ==========================================
# print("\n🏆 HASIL EVALUASI SETELAH OPTIMASI THRESHOLD 🏆")
# print("-" * 50)
# macro_f1 = f1_score(Y_test, y_pred_optimized, average='macro')
# print(f"🌟 MACRO F1-SCORE FINAL: {macro_f1:.4f}")
# print("-" * 50)
# print(classification_report(Y_test, y_pred_optimized, target_names=['Depression', 'Anxiety', 'Stress']))

⏳ 1. Memuat Data Latih, Data Uji, dan Model Final...

🔍 2. Mencari Threshold Terbaik Menggunakan Data Latih (X_train)...
   -> RISK_DEPRESSION: Threshold Kunci = 0.41 (Estimasi F1 Train: 0.7922)
   -> RISK_ANXIETY: Threshold Kunci = 0.37 (Estimasi F1 Train: 0.7876)
   -> RISK_STRESS: Threshold Kunci = 0.38 (Estimasi F1 Train: 0.7858)

🏆 3. Menerapkan Threshold Kunci pada Data Uji Murni (X_test) 🏆
--------------------------------------------------
🌟 MACRO F1-SCORE FINAL (LEGAL): 0.8163
--------------------------------------------------
              precision    recall  f1-score   support

  Depression       0.85      0.77      0.81      3842
     Anxiety       0.86      0.82      0.84      3995
      Stress       0.81      0.79      0.80      3304

   micro avg       0.84      0.79      0.82     11141
   macro avg       0.84      0.79      0.82     11141
weighted avg       0.84      0.79      0.82     11141
 samples avg       0.59      0.63      0.59     11141



d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average